In [1]:
print(" hello world")

 hello world


In [26]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
import itertools
import random
import pandas as pd
import numpy as np
import requests
from uuid import uuid4

In [10]:
from pinecone import ServerlessSpec

index_name = "pinecone-data"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("✅ Index created (or already exists)")

✅ Index created (or already exists)


In [9]:
print(pc.list_indexes())

IndexList([<name='pinecone-data', dim=1536, ready=True>])


In [8]:
index = pc.Index("pinecone-data")

print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1536, total_vector_count=29, metric='cosine', namespaces=1)


In [6]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
pinecone_api_key = os.getenv("PINECONE_API_KEY")

client = OpenAI(api_key=openai_api_key)
pc = Pinecone(api_key=pinecone_api_key)

In [15]:
url = "https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json"

data = requests.get(url).json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['version', 'data'])


In [16]:
records = []

for article in data["data"]:
    title = article["title"]

    for para in article["paragraphs"]:
        context = para["context"]

        records.append({
            "text_id": str(uuid4()),
            "title": title,
            "text": context
        })

df = pd.DataFrame(records)

df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print("Total documents:", len(df))
df.head()

Total documents: 19029


,text_id,title,text
0,16a8d0c8-5412-46ed-ae2c-03b0540d757b,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...
1,dbe7acb7-7ed3-4f00-92fe-83b10219855c,Beyoncé,Following the disbandment of Destiny's Child i...
2,317a3ccb-beaf-48c2-a938-461f5e14af04,Beyoncé,"A self-described ""modern-day feminist"", Beyonc..."
3,140331b4-e2de-479e-b7c1-4d2b0cf2b0cc,Beyoncé,"Beyoncé Giselle Knowles was born in Houston, T..."
4,174e5047-b449-4dbd-af6f-9b09680c62e1,Beyoncé,Beyoncé attended St. Mary's Elementary School ...


In [17]:
batch_size = 100

batches = [
    df[i:i + batch_size]
    for i in range(0, len(df), batch_size)
]

print(f"Total batches: {len(batches)}")
print(f"First batch size: {len(batches[0])}")
print(f"Last batch size: {len(batches[-1])}")

Total batches: 191
First batch size: 100
Last batch size: 29


In [18]:
batch_size = 100

batches = [df[i:i+batch_size] for i in range(0, len(df), batch_size)]

In [19]:
for batch in batches:

    ids = [str(uuid4()) for _ in range(len(batch))]

    texts = batch["text"].tolist()

    metadata = [
        {
            "title": row["title"],
            "text": row["text"]
        }
        for _, row in batch.iterrows()
    ]

In [20]:
response = client.embeddings.create(
    input=texts,
    model="text-embedding-3-small"
)

vectors = [r.embedding for r in response.data]

In [21]:
index.upsert(
    vectors=list(zip(ids, vectors, metadata)),
    namespace="squad_rag_dataset"
)

UpsertResponse(upserted_count=29)

In [22]:
def retrieve_docs(query):

    query_embedding = client.embeddings.create(
        input=query,
        model="text-embedding-3-small"
    ).data[0].embedding

    results = index.query(
        vector=query_embedding,
        top_k=3,
        include_metadata=True,
        namespace="squad_rag_dataset"
    )

    retrieved_docs = []
    sources = []

    for match in results["matches"]:
        retrieved_docs.append(match["metadata"]["text"])
        sources.append(match["metadata"]["title"])

    return retrieved_docs, sources

In [23]:
def build_prompt(query, docs):

    prompt_start = "Answer the question using the context below:\n\n"

    context = "\n\n".join(docs)

    prompt_end = f"\n\nQuestion:{query}"

    return prompt_start + context + prompt_end

In [24]:
def question_answering(prompt, sources):

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    return {
        "answer": answer,
        "sources": sources
    }

In [25]:
query = "What is machine learning?"

docs, sources = retrieve_docs(query)

prompt = build_prompt(query, docs)

result = question_answering(prompt, sources)

print("Answer:\n", result["answer"])
print("\nSources:", result["sources"])

Answer:
 The provided context does not include information about machine learning. To answer your question based on my knowledge, machine learning is a subset of artificial intelligence (AI) that involves the development of algorithms and statistical models that enable computers to perform specific tasks without explicit instructions. Instead, machine learning systems learn from data by identifying patterns and making decisions based on that data. Machine learning can be used for a variety of applications, including image recognition, natural language processing, and predictive analytics.

Sources: ['Matter', 'Matter', 'Matter']
